In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.4 The Stern–Gerlach Experiment and the Birth of the Qubit

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.4",
    title="The Stern–Gerlach Experiment and the Birth of the Qubit",
    blurb="The physics begins. A beam of atoms through a magnetic field splits in "
    "two — and a second, rotated magnet reveals that the first measurement has been "
    "undone. No classical picture survives this. What does is the apparatus we just "
    "built: a state is a vector in two complex dimensions, 'up along x' is a "
    "superposition of up and down along z, and measurement projects. The two-state "
    "system born here is the spin, and the qubit.",
    difficulty="intermediate",
    estimate="140–170 min",
)

## Notebook overview

Three notebooks of mathematics are behind us; this is the first of physics. And it begins, as the
subject historically did, with a single experiment so simple it can be drawn in one line and so
strange it forces the entire quantum framework into existence. A beam of silver atoms is sent
through an inhomogeneous magnetic field. Classically the beam should smear into a continuous band.
It does not: it splits into exactly **two** spots. From that one fact, and a second magnet rotated
relative to the first, every counter-intuitive feature of quantum mechanics follows — quantization,
superposition, the destructiveness of measurement, and the incompatibility of different observables.

The strategy of this notebook is to let the experiment do the forcing. We will not assume the rules
of quantum mechanics and apply them; we will watch the data refuse every classical alternative until
only one account is left standing — the one we built in Movement 0. A spin state is a **vector in
$\mathbb{C}^2$**; the atom that is "up along $x$" is a **superposition** of up and down along $z$,
not a classical mixture and not a hidden label; and a measurement **projects** the state onto the
outcome it finds. The two-state system that emerges is at once the simplest quantum system, the
electron **spin**, and the **qubit** of quantum information — and because its Hilbert space is just
$\mathbb{C}^2$, every computation here is a $2\times2$ matter the formalism handles exactly.

A word on level and honesty. We use two rules in this notebook — that the probability of an outcome
is $|\langle e|\psi\rangle|^2$ (the Born rule), and that the state after a measurement is the
eigenstate found (projection). We use them here because *they make the experiment come out right*,
motivated entirely by the data; [§6.5](postulates.ipynb) elevates them to the **postulates** of quantum mechanics, stated
as axioms. The spin operators $S=(\hbar/2)\,\boldsymbol\sigma$ are previewed lightly — their full
algebra and the uncertainty relation are [§6.6](pauli-uncertainty.ipynb) — and the curious **half-angle** $\theta/2$ that
appears throughout is flagged as the spin-$\tfrac12$ signature behind the Bloch sphere ([§6.8](bloch-sphere-entanglement.ipynb)), noted
but not developed.

Each exercise, as throughout Volume VI, opens with a **crystal-clear statement** and enumerated parts that name the exact operation — explicit two-component complex states, `numpy.vdot`
for amplitudes, `numpy.abs(...)**2` for probabilities, `numpy.random.default_rng` for Born-rule
sampling. Here the mathematics is the easiest of the volume ($2\times2$); the difficulty is purely
conceptual, and it is guided with care, step by step.

> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the two spots as orthonormal spin eigenstates; the half-angle state and the mutually-unbiased
> bases ($|\langle{+}z|{+}x\rangle|^2=\tfrac12$); the projection law $\cos^2(\theta/2)$; the
> sequential $z\to x\to z$ chain resetting to $50/50$; superposition and mixture agreeing on $z$ but
> differing on $x$; and Monte-Carlo Born sampling matching the predicted frequencies. A ✓ is strong
> evidence; a ✗ is a prompt to *locate the discrepancy*.
>
> **Conventions and scope.** We work in the $z$-basis $\{|{+}z\rangle,|{-}z\rangle\}$ with $\hbar=1$,
> so $S_z=\tfrac12\sigma_z$ has eigenvalues $\pm\tfrac12$ (physically $\pm\hbar/2$). The Born rule and
> projection are used here as the rules that fit the experiment; the formal postulates are [§6.5](postulates.ipynb), the
> Pauli algebra and uncertainty are [§6.6](pauli-uncertainty.ipynb), the dynamics is [§6.7](time-evolution.ipynb), and the Bloch-sphere geometry of the
> half-angle is [§6.8](bloch-sphere-entanglement.ipynb). See Sakurai & Napolitano (§1.1); the Feynman Lectures Vol. III; and Notebooks
> [§6.1](complex-vector-spaces.ipynb) (states/amplitudes), [§6.2](operators-spectral-theorem.ipynb) (observables), [§6.3](dirac-notation-spectral-decomposition.ipynb) (projectors).

## Theory in brief

### The experiment and spatial quantization

Silver atoms (one unpaired electron, a magnetic moment $\propto$ spin) pass through a magnetic field
with a gradient along $z$. A classical moment would deflect by a continuous amount; instead the beam
splits into exactly **two** spots. The observable $S_z$ is **quantized**, with two values, and its
eigenstates label an orthonormal basis of a two-dimensional Hilbert space,

```{math}
:label: eq-sg-split
S_z\,|{\pm}z\rangle=\pm\tfrac{\hbar}{2}\,|{\pm}z\rangle,\qquad |{+}z\rangle=\begin{pmatrix}1\\0\end{pmatrix},\ |{-}z\rangle=\begin{pmatrix}0\\1\end{pmatrix} .
```

### States along any axis

An apparatus oriented along the direction $(\theta,\varphi)$ measures spin along that axis, with "+"
eigenstate

```{math}
:label: eq-spin-axis
|{+}\hat n\rangle=\cos\tfrac{\theta}{2}\,|{+}z\rangle+e^{i\varphi}\sin\tfrac{\theta}{2}\,|{-}z\rangle .
```

Note the **half-angle** $\theta/2$ — a spin-$\tfrac12$ signature, the reason a $360^\circ$ rotation
is not the identity (the Bloch sphere and the double cover, [§6.8](bloch-sphere-entanglement.ipynb)). The $z$, $x$, and $y$ bases are
**mutually unbiased**: an atom definite along one axis is exactly $50/50$ along a perpendicular one,
$|\langle{+}z|{+}x\rangle|^2=\tfrac12$.

### The projection (Born) rule, motivated by the data

An atom prepared in $|\psi\rangle$ and measured along an axis with "+" eigenstate $|{+}\hat n\rangle$
comes out "+" with probability

```{math}
:label: eq-born-sg
P(+)=|\langle{+}\hat n|\psi\rangle|^2,\qquad \text{for }|\psi\rangle=|{+}z\rangle:\ P(+)=\cos^2\tfrac{\theta}{2} .
```

After the measurement the atom is **projected** into the eigenstate found — measurement is
destructive. We use this because it reproduces the experiment; [§6.5](postulates.ipynb) makes it the Born postulate.

### The sequential Stern–Gerlach puzzle

Chain three apparatuses. Select $|{+}z\rangle$; measure along $x$ (the atom emerges $\pm x$ with
probability $\tfrac12$ each); keep $|{+}x\rangle$; measure $z$ again — and the atom is now $50/50$,
**not** $100\%$ up,

```{math}
:label: eq-sequential
|{+}z\rangle\xrightarrow{\ x\ }|{+}x\rangle\xrightarrow{\ z\ }\Big\{\,P({+}z)=\tfrac12,\ P({-}z)=\tfrac12\,\Big\} .
```

The intervening $x$-measurement has **destroyed** the original $z$-information. No classical model in
which each atom secretly carries definite $z$- and $x$-labels can reproduce this. This is
**incompatibility**, made experimental (the algebra is [§6.6](pauli-uncertainty.ipynb)).

### Superposition is not a classical mixture

The $|{+}x\rangle$ atom is the **superposition** $(|{+}z\rangle+|{-}z\rangle)/\sqrt2$ — a definite
state of $S_x$, not an ignorance-mixture. The test:

```{math}
:label: eq-sg-superposition
\text{pure }|{+}x\rangle:\ P({+}x)=1,\qquad \text{50/50 mixture of }|{\pm}z\rangle:\ P({+}x)=\tfrac12 ,
```

with **identical** $z$-statistics. Superposition carries a phase a mixture lacks — the seed of the
density matrix ([§6.26](density-matrix.ipynb)).

### The qubit

The two-state system born here {eq}`eq-qubit` is at once the electron **spin** and the **qubit**.
Everything in Movement I lives in this $\mathbb{C}^2$ setting; its geometry is the Bloch sphere ([§6.8](bloch-sphere-entanglement.ipynb)).

```{math}
:label: eq-qubit
|\psi\rangle=\alpha|{+}z\rangle+\beta|{-}z\rangle\in\mathbb{C}^2,\qquad |\alpha|^2+|\beta|^2=1 .
```

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

ACCENT, INK, SOFT, PANEL = draw.ACCENT, draw.INK, draw.SOFT, draw.PANEL

# Conventions: spin states are 2-component complex arrays in the z-basis {|+z⟩, |−z⟩}; we
# set ℏ=1, so S_z = ½σ_z has eigenvalues ±½ (physically ±ℏ/2). Amplitudes are numpy.vdot(plus, state)
# (conjugate-first, §6.1); probabilities are numpy.abs(amplitude)**2; Born-rule sampling uses
# numpy.random.default_rng.

KET_UP = np.array([1, 0], dtype=complex)  # |+z⟩
KET_DOWN = np.array([0, 1], dtype=complex)  # |−z⟩


def spin_state(theta, phi=0.0):
    """The "+" spin eigenstate along the axis $(\\theta,\\varphi)$ {eq}`eq-spin-axis`.

    Returns $|{+}\\hat n\\rangle=\\cos\\tfrac{\\theta}{2}\\,|{+}z\\rangle+e^{i\\varphi}\\sin\\tfrac{
    \\theta}{2}\\,|{-}z\\rangle$, a unit vector in $\\mathbb{C}^2$. The **half-angle** $\\theta/2$ is the
    spin-$\\tfrac12$ signature behind the Bloch sphere (§6.8). With $\\theta=\\pi/2,\\varphi=0$ this is
    $|{+}x\\rangle$; with $\\theta=\\pi/2,\\varphi=\\pi/2$ it is $|{+}y\\rangle$.

    Parameters
    ----------
    theta : float
        Polar angle of the measurement axis from $+z$ (radians).
    phi : float, optional
        Azimuthal angle (radians).

    Returns
    -------
    numpy.ndarray
        The 2-component complex spin state.
    """
    return np.array([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])


def prob(plus, state):
    """Born-rule probability $P(+)=|\\langle\\text{plus}|\\text{state}\\rangle|^2$ {eq}`eq-born-sg`.

    The amplitude is ``numpy.vdot(plus, state)`` (conjugating the first argument, the physics
    convention of §6.1); the probability is its squared magnitude, ``numpy.abs(amplitude)**2``.

    Parameters
    ----------
    plus : numpy.ndarray
        The "+" eigenstate of the measured axis.
    state : numpy.ndarray
        The (normalized) state being measured.

    Returns
    -------
    float
        The probability of the "+" outcome.
    """
    return np.abs(np.vdot(plus, state)) ** 2


def measure(state, plus, n, rng):
    """Sample ``n`` Born-rule outcomes of measuring ``state`` along an axis with "+" eigenstate ``plus``.

    Each atom yields "+" with probability $p=|\\langle\\text{plus}|\\text{state}\\rangle|^2$ and "−"
    otherwise (drawing uniform deviates with ``rng.random`` and comparing to $p$). After the
    measurement the atom is **projected** into the eigenstate found — the post-measurement state is
    ``plus`` on a "+" outcome and its orthogonal partner on a "−" outcome. We return the empirical "+"
    frequency together with the exact probability for comparison.

    Parameters
    ----------
    state : numpy.ndarray
        The state entering the apparatus.
    plus : numpy.ndarray
        The "+" eigenstate of the measured axis.
    n : int
        Number of atoms to send through.
    rng : numpy.random.Generator
        A ``numpy.random.default_rng`` instance.

    Returns
    -------
    frequency : float
        The empirical fraction of "+" outcomes.
    p : float
        The exact Born probability $|\\langle\\text{plus}|\\text{state}\\rangle|^2$.
    """
    p = prob(plus, state)
    outcomes = rng.random(n) < p  # True = "+" outcome
    return outcomes.mean(), p


# the named axis states we use throughout
KET_X_PLUS = spin_state(np.pi / 2, 0.0)  # |+x⟩ = (|+z⟩ + |−z⟩)/√2
KET_X_MINUS = spin_state(np.pi / 2, np.pi)  # |−x⟩ = (|+z⟩ − |−z⟩)/√2
KET_Y_PLUS = spin_state(np.pi / 2, np.pi / 2)  # |+y⟩

## Exercise 1 — The two-state system and spatial quantization

The Stern–Gerlach beam splits into exactly two spots. Write the two spin states that label them,
confirm they form an orthonormal basis of $\mathbb{C}^2$, and verify they are the eigenstates of
the spin observable $S_z=\tfrac12\sigma_z$ with eigenvalues $\pm\tfrac12$ (physically
$\pm\hbar/2$) {eq}`eq-sg-split`.

1. Write $|{+}z\rangle$ and $|{-}z\rangle$ as the standard basis vectors (the 2-component complex
   arrays `KET_UP`, `KET_DOWN`).
2. Verify orthonormality: $\langle{+}z|{-}z\rangle =0$ and unit norms, with `numpy.vdot`.
3. Build $S_z=\tfrac12\sigma_z$ (a $2\times2$ array) and confirm with `numpy.allclose` that `S_z @
   KET_UP` equals $+\tfrac12\,$`KET_UP` and `S_z @ KET_DOWN` equals $-\tfrac12\,$`KET_DOWN` — the
   two spots are its eigenstates (a callback to [§6.2](operators-spectral-theorem.ipynb): a Hermitian observable's eigenvalues are the
   possible outcomes).

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    orthonormal and up_is_eig and down_is_eig,
    "the two beam spots are the orthonormal eigenstates |±z⟩ of the spin observable S_z, with eigenvalues ±ℏ/2",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — Spin along an arbitrary axis

Construct the "+" eigenstate of spin measured along the direction $(\theta,\varphi)$, confirm it
is normalized, and verify the **mutually-unbiased** property of the $z$- and $x$-bases: an atom
definite along $z$ is exactly $50/50$ along $x$, $|\langle{+}z|{+}x\rangle|^2=\tfrac12$
{eq}`eq-spin-axis`.

1. Build $|{+}\hat n\rangle=\cos\tfrac{\theta}{2}|{+}z\rangle+e^{i\varphi}\sin\tfrac{
   \theta}{2}|{-}z\rangle$ with the `spin_state` helper (note the half-angle $\theta/2$).
2. Confirm it is normalized, $\langle{+}\hat n|{+}\hat n\rangle=1$, with `numpy.vdot`.
3. Form $|{+}x\rangle=$ `spin_state(numpy.pi/2, 0)` and compute $|\langle{+}z|{+}x\rangle|^2$ with
   the `prob` helper (`numpy.abs(numpy.vdot(...))**2`); confirm it is $\tfrac12$.
4. Check the same for the $z$–$y$ and $x$–$y$ pairs — all three bases are mutually unbiased.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    np.vdot(n_state, n_state).real,
    1.0,
    "the spin state |+n̂⟩ = cos(θ/2)|+z⟩ + e^{iφ}sin(θ/2)|−z⟩ is normalized",
    rtol=1e-12,
)
validate.close(
    [p_zx, p_zy, p_xy],
    [0.5, 0.5, 0.5],
    "the z-, x-, and y-bases are mutually unbiased — a definite atom along one axis is 50/50 along a perpendicular one",
    rtol=1e-12,
)

## Exercise 3 — The projection law $P(+)=\cos^2(\theta/2)$

A $z$-up atom is measured along an axis tilted by an angle $\theta$ from $z$ (in the $x$–$z$
plane). Find the probability $P(+)$ of the "+" outcome as a function of $\theta$, and verify it
equals $\cos^2(\theta/2)$ — the Born rule read straight off the experiment {eq}`eq-born-sg`.

1. For each $\theta$, build the tilted "+" eigenstate `spin_state(theta, 0)`.
2. Compute $P(+)=|\langle{+}\hat n(\theta)|{+}z\rangle|^2$ with the `prob` helper over a grid of
   $\theta$ from `numpy.linspace(0, numpy.pi, ...)`.
3. Compare against `numpy.cos(theta/2)**2` with `numpy.allclose`, and read off the landmarks: $1$
   at $\theta=0$ (same axis, certain), $\tfrac12$ at $\theta=\pi/2$ (perpendicular, $50/50$), $0$
   at $\theta=\pi$ (opposite, never). Frame: this is the Born rule, motivated by the data; [§6.5](postulates.ipynb)
   makes it a postulate.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    P_plus,
    np.cos(theta_grid / 2) ** 2,
    "the projection probability for a z-up atom measured along a tilted axis is cos²(θ/2)",
    rtol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The sequential Stern–Gerlach experiment

Send a $z$-up atom through an $x$-apparatus, keep only the $|{+}x\rangle$ output, then measure $z$
again. Find the final $z$-distribution, and contrast it with the case of no intervening
$x$-measurement ($z\to z$ directly). The intervening measurement resets the $z$-distribution to
$50/50$, whereas $z\to z$ stays $100\%$ up {eq}`eq-sequential`.

1. Start in $|{+}z\rangle$ (`KET_UP`).
2. Measure along $x$: compute $P({+}x)=$ `prob(KET_X_PLUS, KET_UP)` (it is $\tfrac12$), and
   **project** onto the kept output — the state becomes $|{+}x\rangle$ (`KET_X_PLUS`).
3. Measure along $z$ on that projected state: compute $P({+}z)=$`prob(KET_UP, KET_X_PLUS)` and
   $P({-}z)=$`prob(KET_DOWN, KET_X_PLUS)` — find $50/50$.
4. Contrast with measuring $z$ directly on $|{+}z\rangle$: $P({+}z)=$`prob(KET_UP, KET_UP)` $=1$.
   Frame: the intervening $x$-measurement has destroyed the original $z$-information —
   incompatibility, made experimental. No hidden-label model survives.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    [P_plusz_after_x, P_plusz_direct],
    [0.5, 1.0],
    "an intervening x-measurement resets the z-distribution to 50/50, whereas z→z stays 100% up — measurement disturbs incompatible observables",
    rtol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Superposition versus classical mixture

Distinguish the pure superposition $|{+}x\rangle=(|{+}z\rangle+|{-}z\rangle)/\sqrt2$ from a
$50/50$ **classical mixture** of $|{+}z\rangle$ and $|{-}z\rangle$ by their measurement
statistics. Show they agree perfectly on $z$ but disagree on $x$ — so "up along $x$" is a definite
state, not ignorance {eq}`eq-sg-superposition`.

1. For the pure $|{+}x\rangle$ (`KET_X_PLUS`), compute the $z$-statistics
   $P({\pm}z)=$`prob(KET_UP/KET_DOWN, KET_X_PLUS)`. For the mixture, the $z$-statistics are
   $\tfrac12, \tfrac12$ by construction (half the atoms are $|{+}z\rangle$, half $|{-}z\rangle$).
   Show they match.
2. Compute the $x$-statistics. For the pure state, $P({+}x)=$`prob(KET_X_PLUS, KET_X_PLUS)` $=1$.
   For the mixture, average the two branches: $P({+}x)=\tfrac12\,$`prob(KET_X_PLUS,
   KET_UP)`$+\tfrac12\,$ `prob(KET_X_PLUS, KET_DOWN)` $=\tfrac12$.
3. Conclude: identical on $z$, different on $x$ — the superposition carries a relative phase the
   mixture lacks (the seed of the density matrix, [§6.26](density-matrix.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    [P_plusx_pure, P_plusx_mixture],
    [1.0, 0.5],
    "a superposition and a classical mixture share z-statistics but differ on x (P(+x)=1 vs ½) — superposition is not ignorance",
    rtol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Simulating the apparatus: Born-rule Monte Carlo

We have *computed* the Stern–Gerlach probabilities; now **run** the experiment. Send many atoms
through a measurement by sampling each outcome from the Born rule, and confirm the empirical
frequencies match the predicted probabilities for the $z\to x$ and $x\to z$ steps
{eq}`eq-born-sg`.

1. Create a generator with `numpy.random.default_rng(seed)`.
2. For the $z\to x$ step (a $|{+}z\rangle$ atom measured along $x$), call the `measure` helper,
   which draws $n$ outcomes with `rng.random(n) < p` and returns the empirical "+" frequency
   alongside the exact $p=|\langle{+}x|{+}z \rangle|^2$.
3. Repeat for the $x\to z$ step (a $|{+}x\rangle$ atom measured along $z$).
4. Confirm each frequency matches its probability to within the sampling error $\sim 1/\sqrt{n}$.
   Frame: we can *run* the experiment, not just solve it — the Monte-Carlo spine of Volume V, in a
   quantum setting.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    [freq_zx, freq_xz],
    [p_zx, p_xz],
    "Monte-Carlo sampling of the Born rule reproduces the predicted Stern–Gerlach frequencies",
    atol=0.01,
)

## Exercise 7 — A three-axis chain and the cost of looking *(student)*

Two parts. **(a)** Prepare $|{+}z\rangle$ and compare two protocols for how much "$z$-up"
survives: protocol **A** measures $z$ directly; protocol **B** measures $x$, then $z$. **(b)**
Then show that measurement can *steer* a state: design a chain of $N$ equally-spaced rotated
measurements that carries $|{+}z\rangle$ all the way to $|{-}z\rangle$, and show the success
probability approaches $1$ as $N$ grows {eq}`eq-sequential`, {eq}`eq-born-sg`.

1. Protocol A: $P({+}z)=$`prob(KET_UP, KET_UP)` $=1$. Protocol B: measure $x$ (project to
   $|{+}x\rangle$), then $z$: $P({+}z)=$`prob(KET_UP, KET_X_PLUS)` $=\tfrac12$. Compare — B has
   destroyed half the $z$-up information A preserved.
2. For the steering chain, tilt the measurement axis in $N$ equal steps from $\theta=0$ to
   $\theta=\pi$; each step rotates the axis by $\Delta\theta= \pi/N$, so each projection succeeds
   with probability $\cos^2(\Delta\theta/2)$ (the law of Exercise 3). The probability of surviving
   all $N$ projections — ending in $|{-}z\rangle$ — is $[\cos^2(\pi/2N)]^N$.
3. Evaluate this with `numpy.cos` for a range of $N$ (e.g. `numpy.array([1, 2, 5, 20, 100])`) and
   confirm it rises toward $1$. Frame: measurement is not passive — a sequence of gentle
   measurements can drag the state from up to down (a first glimpse of the quantum Zeno idea).

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    np.isclose(P_A, 1.0) and np.isclose(P_B, 0.5),
    "protocol B (an intervening x-measurement) destroys z-up information that protocol A (z direct) preserves",
)
validate.check(
    P_drag[0] < 0.01 and P_drag[-1] > 0.97 and np.all(np.diff(P_drag) > 0),
    "a chain of N gentle rotated measurements steers |+z⟩ toward |−z⟩, with success → 1 as N grows — measurement actively changes the state",
)

## Exercise 8 — One experiment, the whole theory

A beam splitting in two told us that spin is **quantized** — the observable $S_z$ has two values, and
its eigenstates span a two-dimensional Hilbert space. A second, rotated magnet told us that
measurements of different spin components are **incompatible**: looking along $x$ erases what we knew
along $z$, and no model in which each atom secretly carries definite labels can reproduce the
$50/50$ reset. And the only account that fits is the one Movement 0 built — a state is a **vector in
$\mathbb{C}^2$**, "up along $x$" is a **superposition** of up and down along $z$ (not a mixture, not
a hidden label, as its $x$-statistics prove), and measurement **projects**. The two-state system we
have met is the electron spin and the qubit, and it is the arena for all of Movement I.

**This exercise (synthesis).** No new computation: the lesson is the result. We did not assume quantum
mechanics and apply it — we watched an experiment refuse every classical alternative and back us into
it. The strangeness is not an artefact of the formalism; it is in the data, and the formalism is
simply what survives. The next notebook ([§6.5](postulates.ipynb)) takes the two rules we used here — outcomes drawn with
probability $|\langle e|\psi\rangle|^2$, and the state projected onto the outcome found — and states
them, with the rest, as the **postulates** of quantum mechanics, once and for all.

## Notebook summary

The first physics of Volume VI: one experiment that forces the whole framework, in the smallest
Hilbert space.

- **Spatial quantization** {eq}`eq-sg-split`: the beam splits into two, so $S_z$ is quantized; the
  two spots are the orthonormal eigenstates $|{\pm}z\rangle$ (eigenvalues $\pm\hbar/2$).
- **States along any axis** {eq}`eq-spin-axis`: $|{+}\hat n\rangle=\cos\tfrac{\theta}{2}|{+}z\rangle+
  e^{i\varphi}\sin\tfrac{\theta}{2}|{-}z\rangle$, with the spin-$\tfrac12$ half-angle; $z,x,y$ are
  mutually unbiased ($|\langle{+}z|{+}x\rangle|^2=\tfrac12$).
- **The projection (Born) rule** {eq}`eq-born-sg`: $P(+)=|\langle{+}\hat n|\psi\rangle|^2$, giving
  $\cos^2(\theta/2)$ for a $z$-up atom — the rule that fits the data (the postulate is [§6.5](postulates.ipynb)).
- **The sequential puzzle** {eq}`eq-sequential`: $z\to x\to z$ resets to $50/50$ while $z\to z$ stays
  definite — measurement disturbs incompatible observables. No hidden-label model survives.
- **Superposition $\ne$ mixture** {eq}`eq-sg-superposition`: identical $z$-statistics, $P({+}x)=1$ vs
  $\tfrac12$ on $x$ — superposition carries a phase ignorance lacks (the density matrix, [§6.26](density-matrix.ipynb)).
- **Born-rule Monte Carlo**: sampling outcomes with `numpy.random.default_rng` reproduces the
  predicted frequencies — we can *run* the apparatus, not just solve it.

The two-state system born here is the spin and the qubit. We did not assume quantum mechanics; the
experiment forced it. The next notebook makes the rules we used into postulates.

## Outlook

- **The postulates of quantum mechanics ([§6.5](postulates.ipynb))**: states, observables, the Born rule, and
  measurement/projection, stated as axioms — the rules of this notebook made formal.
- **The Pauli algebra and the uncertainty relation ([§6.6](pauli-uncertainty.ipynb))**: the commutators behind the
  incompatibility seen here, and the bound it implies.
- **Time evolution ([§6.7](time-evolution.ipynb))**: spin precession and Rabi oscillations, the qubit set in motion.
- **The Bloch sphere ([§6.8](bloch-sphere-entanglement.ipynb))**: the geometry of the qubit, where the half-angle $\theta/2$ becomes a
  point on a sphere and the double cover is made visible.
- **Superposition versus mixture, made formal: the density matrix ([§6.26](density-matrix.ipynb)).**
- **Cross-reference** [§6.1](complex-vector-spaces.ipynb) (states/amplitudes), [§6.2](operators-spectral-theorem.ipynb) (observables/eigenvalues), [§6.3](dirac-notation-spectral-decomposition.ipynb) (projectors), and
  forward to [§6.5](postulates.ipynb), [§6.6](pauli-uncertainty.ipynb), [§6.7](time-evolution.ipynb), [§6.8](bloch-sphere-entanglement.ipynb), [§6.26](density-matrix.ipynb).

In [ ]:
from ecp.style import footer

footer()